# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa
This notebook demonstrates how to load, explore, and analyze the FAIR² Mental Health Survey Dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema and is accessible at:
- [https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")
print(f"Keywords: {', '.join(metadata.keywords)}")

## 2. Data Overview
Review available record sets, fields, their IDs and overview each component as referenced by their `@id`.

In [ ]:
# List dataset record sets with their @id
record_sets = list(dataset.record_sets())
print("Record Sets available:")
for rs in record_sets:
    print(f"- Name: {rs.name}")
    print(f"  @id: {rs['@id']}")
    print(f"  Description: {getattr(rs, 'description', 'No description')}")
    print(f"  Fields:")
    for f in rs.fields:
        print(f"    - Field name: {f.name}")
        print(f"      @id: {f['@id']}")
        print(f"      Data type: {getattr(f, 'dataType', 'unknown')}")
    print("---")

# For demonstration, preview records from the first record set
if record_sets:
    preview_rs_id = record_sets[0]['@id']
    print(f"Previewing records from record set '@id': {preview_rs_id}")
    for i, record in enumerate(dataset.records(record_set=preview_rs_id)):
        print(record)
        if i > 2:
            break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the `@id`s as discovered in the overview.

In [ ]:
# Extract data from each record set
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]
for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df

# Show columns and preview from the first record set
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"Columns for record set @id '{first_rs_id}':")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filter records, normalize numeric columns, categorize and group. All actions reference fields by their `@id`.

In [ ]:
# EDA example using the PHQ-9 score (depression scale)
# Identify the numeric field by @id
phq9_field_id = 'phq9_score' # Replace by actual field @id after inspection
rs_id = first_rs_id

# Check if PHQ-9 score exists
if phq9_field_id in dataframes[rs_id].columns:
    df = dataframes[rs_id]
    threshold = 10  # Moderate+ symptoms threshold
    filtered_df = df[df[phq9_field_id] > threshold]
    print(f"Filtered records with {phq9_field_id} > {threshold}:")
    print(filtered_df.head())
    
    # Normalize PHQ-9 score
    norm_col = f"{phq9_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[phq9_field_id] - filtered_df[phq9_field_id].mean()) / filtered_df[phq9_field_id].std()
    print(f"Normalized {phq9_field_id} for filtered records:")
    print(filtered_df[[phq9_field_id, norm_col]].head())
    
    # Group by education, if present
    education_field_id = 'level_of_education'  # Use actual @id
    if education_field_id in df.columns:
        grouped = filtered_df.groupby(education_field_id)[phq9_field_id].mean().reset_index()
        print(f"Grouped means of PHQ-9 score by {education_field_id}:")
        print(grouped.head())
else:
    print(f"Field '{phq9_field_id}' not found in record set '{rs_id}'.")

## 5. Visualization
Visualize the distribution of PHQ-9 scores and relationships by education group.
All axes/labels must reference specific fields by their `@id` where possible.

In [ ]:
# Visualize PHQ-9 score distributions
import matplotlib.pyplot as plt
import seaborn as sns

if phq9_field_id in dataframes[rs_id].columns:
    fig, ax = plt.subplots(figsize=(8,5))
    sns.histplot(dataframes[rs_id][phq9_field_id], bins=15, kde=True, ax=ax)
    ax.set_title(f"Distribution of PHQ-9 scores (@id: {phq9_field_id})")
    ax.set_xlabel(f"PHQ-9 Score (@id: {phq9_field_id})")
    ax.set_ylabel("Frequency")
    plt.show()
    
    # Boxplot of PHQ-9 score by education
    if education_field_id in dataframes[rs_id].columns:
        fig, ax = plt.subplots(figsize=(10,6))
        sns.boxplot(x=education_field_id, y=phq9_field_id, data=dataframes[rs_id], ax=ax)
        ax.set_title(f"PHQ-9 Scores by Education (@id: {education_field_id})")
        ax.set_xlabel(f"Education Level (@id: {education_field_id})")
        ax.set_ylabel(f"PHQ-9 Score (@id: {phq9_field_id})")
        plt.show()
else:
    print(f"Visualization skipped. Field '{phq9_field_id}' not found.")

## 6. Conclusion
This notebook demonstrated loading and processing the FAIR² Mental Health Survey Dataset using `mlcroissant`. We:
- Loaded the dataset and listed record sets and their fields by `@id`.
- Extracted tabular data for analysis.
- Filtered and normalized PHQ-9 scores, and grouped records by education level.
- Visualized distributions and relationships in the data.

This workflow can be adapted for other Croissant datasets by referencing entities by their `@id` throughout.

**Next steps:**
- Apply further statistical tests.
- Use additional record sets or fields for advanced modeling.
- Integrate this data with other FAIR datasets for broader insights.